# Negotiating the Past - Full Dataset Classification

Run MLX classification on the entire dataset (~10.7M prompts).
No visualization — just process and save results with checkpointing.

**Robustness features:**
- Append-mode CSV writing (no full rewrite at each checkpoint)
- Consecutive error detection — auto-stops if the model crashes
- KeyboardInterrupt handling — saves progress before exiting
- Guards that verify model is loaded before starting
- Skips empty/junk prompts (single chars, bare URLs, bare numbers)
- Duplicate header detection on resume

## 1. Setup

In [1]:
import os
import time
import logging
import re
import csv
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Configuration
FULL_RESULTS_FILE = "data/results_mlx/full_results.csv"
CHECKPOINT_INTERVAL = 1000  # Flush to disk every N prompts
MAX_CONSECUTIVE_ERRORS = 20  # Stop if this many errors in a row
MIN_PROMPT_LENGTH = 3  # Skip prompts shorter than this

os.makedirs("data/results_mlx", exist_ok=True)
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f"logs/mlx_full_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("mlx_classifier")
print("Libraries loaded")
print(f"Results file: {FULL_RESULTS_FILE}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL}")
print(f"Max consecutive errors before auto-stop: {MAX_CONSECUTIVE_ERRORS}")

/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Libraries loaded
Results file: data/results_mlx/full_results.csv
Checkpoint interval: 1000
Max consecutive errors before auto-stop: 20


## 2. Load Model

In [2]:
MODEL_ID = "mlx-community/Ministral-3-3B-Instruct-2512-4bit"

from mlx_lm import load, generate
from huggingface_hub import snapshot_download

print(f"Model: {MODEL_ID}")
print("Downloading model (if not cached)...")
local_path = snapshot_download(repo_id=MODEL_ID)
print(f"Model cached at: {local_path}")

print("Loading model into memory...")
model, tokenizer = load(local_path)
print("Model loaded successfully!")

2026-03-18 18:07:01,146 - INFO - HTTP Request: GET https://huggingface.co/api/models/mlx-community/Ministral-3-3B-Instruct-2512-4bit/revision/main "HTTP/1.1 200 OK"


Model: mlx-community/Ministral-3-3B-Instruct-2512-4bit


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Model cached at: /Users/fcl/.cache/huggingface/hub/models--mlx-community--Ministral-3-3B-Instruct-2512-4bit/snapshots/a962dcb09eee4169c890e544c9eb938f1113fdee
Loading model into memory...
Model loaded successfully!


## 3. System Prompt & Classification Function

In [3]:
SYSTEM_PROMPT = """Analyze if this image generation prompt contains a reference to the historical past.

Answer YES if the prompt contains:
- Historical figures (Napoleon, Caesar, Cleopatra, Marie Antoinette, etc.)
- Historical events (World War, Revolution, Cold War, etc.)
- Historical periods or eras (Victorian, Medieval, Renaissance, 1920s, Ancient Rome, etc.)
- Historical artists or their works (Da Vinci, Rembrandt, Michelangelo, etc.)
- Historical art movements (Baroque, Art Nouveau, Impressionism, etc.)
- Mythology and ancient legends (Greek gods, Norse mythology, Egyptian mythology, etc.)

Answer NO if the prompt:
- Only uses stylistic words (vintage, retro, sepia, old photograph)
- Only describes fictional/fantasy content (steampunk, cyberpunk, sci-fi)
- Only mentions living celebrities in modern context
- References extinct animals without historical context (dinosaurs, dodo)
- Is purely futuristic with no past reference

Format: [yes/no]: [one sentence reason]
"""

from mlx_lm.sample_utils import make_sampler
greedy_sampler = make_sampler(temp=0.0)


def format_prompt_for_model(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze this prompt: {user_prompt}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def parse_response(response: str) -> tuple[str, str]:
    response = response.strip()
    match = re.match(r'^(yes|no)\s*[:\-]\s*(.+)', response, re.IGNORECASE | re.DOTALL)
    if match:
        classification = match.group(1).lower()
        justification = match.group(2).strip()
        justification = justification.split('.')[0] + '.' if '.' in justification else justification
        return classification, justification
    response_lower = response.lower()
    if response_lower.startswith('yes'):
        return 'yes', response[3:].strip(' :-')
    elif response_lower.startswith('no'):
        return 'no', response[2:].strip(' :-')
    if 'yes' in response_lower[:50]:
        return 'yes', response
    elif 'no' in response_lower[:50]:
        return 'no', response
    logger.warning(f"Could not parse response: {response[:100]}")
    return 'error', response


def classify_prompt(prompt: str, max_tokens: int = 100) -> tuple[str, str, float]:
    formatted = format_prompt_for_model(prompt)
    start_time = time.time()
    response = generate(model, tokenizer, prompt=formatted, max_tokens=max_tokens, sampler=greedy_sampler)
    generation_time = time.time() - start_time
    classification, justification = parse_response(response)
    return classification, justification, generation_time


def is_valid_prompt(prompt: str) -> bool:
    """Filter out prompts that are too short, bare URLs, or bare numbers."""
    if len(prompt) < MIN_PROMPT_LENGTH:
        return False
    # Bare number
    if prompt.replace('.', '').replace('-', '').isdigit():
        return False
    # Bare URL
    if prompt.startswith(('http://', 'https://', 'www.')):
        return False
    return True




def count_csv_rows(filepath: str) -> int:
    """Count data rows in a CSV file, correctly handling quoted multiline fields."""
    with open(filepath, 'r', newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        count = sum(1 for _ in reader) - 1  # subtract header
    return max(0, count)


print("Classification function defined")
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

Classification function defined
System prompt: 966 chars


## 4. Quick Sanity Check

Verify the model works before launching the full run. This cell **must pass** before proceeding.

In [4]:
# Run 3 test prompts to make sure classify_prompt is fully operational
test_cases = [
    ("Napoleon Bonaparte leading his army across the Alps", "yes"),
    ("a cute cat sitting on a rainbow", "no"),
    ("portrait of Julius Caesar in marble", "yes"),
]

print("Running sanity checks...")
all_passed = True
for test_prompt, expected in test_cases:
    classification, justification, gen_time = classify_prompt(test_prompt)
    status = "OK" if classification == expected else "MISMATCH"
    if classification not in ('yes', 'no'):
        status = "FAIL"
        all_passed = False
    print(f"  [{status}] '{test_prompt[:50]}...' -> {classification} (expected {expected}, {gen_time:.2f}s)")

assert all_passed, "Sanity check FAILED — classify_prompt returned 'error'. Do NOT proceed to full run."
print("\nAll sanity checks passed! Model is operational.")

Running sanity checks...
  [OK] 'Napoleon Bonaparte leading his army across the Alp...' -> yes (expected yes, 1.77s)
  [OK] 'a cute cat sitting on a rainbow...' -> no (expected no, 0.56s)
  [OK] 'portrait of Julius Caesar in marble...' -> yes (expected yes, 0.57s)

All sanity checks passed! Model is operational.


## 5. (Optional) Inspect & Trim Previous Results

Run this cell to check the state of any existing results file.
It will:
- Detect and remove duplicate header rows
- Detect and remove error rows from a crashed run
- Preserve valid rows (yes, no, skipped)
- Back up the original before making changes

In [5]:
if os.path.exists(FULL_RESULTS_FILE):
    df = pd.read_csv(FULL_RESULTS_FILE)
    total_before = len(df)

    # --- Detect duplicate headers embedded as data rows ---
    header_mask = (df['prompt'] == 'prompt') & (df['references_past'] == 'references_past')
    dup_headers = header_mask.sum()

    # --- Count errors ---
    error_mask = df['references_past'] == 'error'
    error_count = error_mask.sum()

    # --- Report ---
    counts = df["references_past"].value_counts()
    print(f"Current file: {total_before:,} rows")
    for val, cnt in counts.items():
        print(f"  {val}: {cnt:,}")
    if dup_headers > 0:
        print(f"  WARNING: {dup_headers} duplicate header row(s) detected")

    # --- Trim if needed ---
    needs_trim = error_count > 0 or dup_headers > 0
    if needs_trim:
        import shutil
        backup = f"data/results_mlx/full_results_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        shutil.copy2(FULL_RESULTS_FILE, backup)
        print(f"\nBackup saved to {backup}")

        # Keep yes, no, AND skipped rows (skipped are valid — just junk prompts)
        df_valid = df[df["references_past"].isin(["yes", "no", "skipped"])].reset_index(drop=True)
        df_valid.to_csv(FULL_RESULTS_FILE, index=False)
        removed = total_before - len(df_valid)
        print(f"Trimmed: {total_before:,} -> {len(df_valid):,} rows ({removed:,} bad rows removed)")
    else:
        print("\nFile is clean — no errors or duplicate headers.")
else:
    print(f"No existing results file at {FULL_RESULTS_FILE}")
    print("Will start from scratch.")

Current file: 10,670,393 rows
  error: 9,494,572
  no: 1,100,547
  yes: 75,274

Backup saved to data/results_mlx/full_results_backup_20260318_180727.csv
Trimmed: 10,670,393 -> 1,175,821 rows (9,494,572 bad rows removed)


## 6. Run Full Dataset

Processes all prompts with checkpointing. **Safe to interrupt and resume.**

Key safety features:
- **Append mode**: writes new results incrementally (no full rewrite)
- **Consecutive error detection**: auto-stops after 20 errors in a row (model probably crashed)
- **KeyboardInterrupt**: catches Ctrl+C and saves buffered results before stopping
- **Prompt validation**: skips empty, too-short, URL-only, and number-only prompts
- **Periodic stats**: logs speed, error rate, and ETA every checkpoint

In [ ]:
# ============================================================
# GUARD: verify everything is defined before starting
# ============================================================
try:
    _ = model, tokenizer, greedy_sampler, classify_prompt, SYSTEM_PROMPT
except NameError as e:
    raise RuntimeError(
        f"Missing required variable: {e}. "
        "Run cells 2-4 first (Load Model, System Prompt, Sanity Check)."
    )

# ============================================================
# LOAD PROMPTS
# ============================================================
print("Loading full dataset...")
full_prompts_df = pd.read_csv('data/prompts.csv', usecols=[0])
full_prompts = full_prompts_df.iloc[:, 0].dropna().tolist()
full_prompts = [str(p).strip() for p in full_prompts if str(p).strip()]
print(f"Total prompts in source: {len(full_prompts):,}")

# ============================================================
# DETERMINE RESUME POINT
# ============================================================
start_idx = 0
if os.path.exists(FULL_RESULTS_FILE) and os.path.getsize(FULL_RESULTS_FILE) > 0:
    # Count rows using csv.reader to correctly handle quoted multiline fields
    start_idx = count_csv_rows(FULL_RESULTS_FILE)
    print(f"Existing results: {start_idx:,} rows")

    # Verify last line is not a duplicate header
    with open(FULL_RESULTS_FILE, 'rb') as f:
        f.seek(max(0, f.seek(0, 2) - 200))  # read last 200 bytes
        last_lines = f.read().decode('utf-8', errors='replace').strip().split('\n')
        last_line = last_lines[-1] if last_lines else ''
    if 'prompt' in last_line and 'references_past' in last_line:
        logger.warning("Last line is a duplicate header — file may be corrupted. Run cell 5 first.")

remaining = len(full_prompts) - start_idx
print(f"Resuming from index: {start_idx:,}")
print(f"Remaining to process: {remaining:,}")

if remaining <= 0:
    print("\nNothing to process — all prompts already classified!")

# ============================================================
# OPEN CSV IN APPEND MODE
# ============================================================
if remaining > 0:
    write_header = not os.path.exists(FULL_RESULTS_FILE) or os.path.getsize(FULL_RESULTS_FILE) == 0
    csv_file = open(FULL_RESULTS_FILE, 'a', newline='', encoding='utf-8')
    csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_ALL)

    if write_header:
        csv_writer.writerow(['prompt', 'references_past', 'justification', 'generation_time'])
        csv_file.flush()

    # ============================================================
    # PROCESSING LOOP
    # ============================================================
    consecutive_errors = 0
    total_errors = 0
    total_skipped = 0
    buffer = []  # buffer rows between flushes
    processed_since_resume = 0
    run_start_time = time.time()

    def flush_buffer():
        """Write buffered rows to CSV and flush to disk."""
        for row in buffer:
            csv_writer.writerow(row)
        csv_file.flush()
        buffer.clear()

    try:
        for i, prompt in enumerate(tqdm(
            full_prompts[start_idx:],
            desc="Processing",
            initial=start_idx,
            total=len(full_prompts),
        )):
            # --- Skip or classify ---
            if not is_valid_prompt(prompt):
                buffer.append([prompt, 'skipped', 'invalid prompt (too short, URL, or number)', 0])
                total_skipped += 1
            else:
                try:
                    classification, justification, gen_time = classify_prompt(prompt)
                    buffer.append([prompt, classification, justification, gen_time])

                    if classification == 'error':
                        consecutive_errors += 1
                        total_errors += 1
                    else:
                        consecutive_errors = 0

                except KeyboardInterrupt:
                    raise  # re-raise to outer handler
                except Exception as e:
                    buffer.append([prompt, 'error', str(e), 0])
                    consecutive_errors += 1
                    total_errors += 1
                    logger.error(f"Error at index {start_idx + i}: {e}")

            processed_since_resume += 1

            # --- Consecutive error detection ---
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                # Save error message BEFORE flushing (flush clears the buffer)
                last_error_msg = buffer[-1][2][:200] if buffer else 'unknown'
                logger.critical(
                    f"STOPPED: {MAX_CONSECUTIVE_ERRORS} consecutive errors detected. "
                    f"Last error: {last_error_msg[:100]}"
                )
                flush_buffer()
                csv_file.close()
                total_now = start_idx + processed_since_resume
                print(f"\n{'!' * 60}")
                print(f"AUTO-STOPPED after {MAX_CONSECUTIVE_ERRORS} consecutive errors!")
                print(f"Saved {total_now:,} total results.")
                print(f"Last error: {last_error_msg}")
                print(f"\nTo resume: fix the issue, run cell 5 to trim errors, then re-run this cell.")
                print(f"{'!' * 60}")
                raise RuntimeError(f"Too many consecutive errors ({MAX_CONSECUTIVE_ERRORS}). Model may have crashed.")

            # --- Checkpoint ---
            if processed_since_resume % CHECKPOINT_INTERVAL == 0:
                flush_buffer()

                elapsed = time.time() - run_start_time
                rate = processed_since_resume / elapsed
                total_now = start_idx + processed_since_resume
                eta_seconds = (len(full_prompts) - total_now) / rate if rate > 0 else 0
                eta_hours = eta_seconds / 3600

                logger.info(
                    f"Checkpoint: {total_now:,}/{len(full_prompts):,} "
                    f"({total_now/len(full_prompts)*100:.1f}%) | "
                    f"{rate:.1f} prompts/sec | "
                    f"errors: {total_errors} | skipped: {total_skipped} | "
                    f"ETA: {eta_hours:.1f}h"
                )

    except KeyboardInterrupt:
        print("\n\nInterrupted by user — saving buffered results...")
        flush_buffer()
        csv_file.close()
        total_now = start_idx + processed_since_resume
        elapsed = time.time() - run_start_time
        print(f"Saved {total_now:,} total results ({processed_since_resume:,} this session, {elapsed/60:.1f} min)")
        print(f"Errors: {total_errors} | Skipped: {total_skipped}")
        print("\nRe-run this cell to resume from where you left off.")
    else:
        # Normal completion
        flush_buffer()
        csv_file.close()
        elapsed = time.time() - run_start_time
        total_now = start_idx + processed_since_resume
        print(f"\n{'=' * 60}")
        print(f"COMPLETE! {total_now:,} prompts classified.")
        print(f"Session: {processed_since_resume:,} prompts in {elapsed/60:.1f} min ({elapsed/3600:.1f}h)")
        print(f"Rate: {processed_since_resume/elapsed:.1f} prompts/sec")
        print(f"Errors: {total_errors} | Skipped: {total_skipped}")
        print(f"Results saved to {FULL_RESULTS_FILE}")
        print(f"{'=' * 60}")

Loading full dataset...
Total prompts in source: 10,670,393
Existing results: 1,175,821 rows
Resuming from index: 1,175,821
Remaining to process: 9,494,572


Processing:  11%|#1        | 1175821/10670393 [00:00<?, ?it/s]

2026-03-18 18:20:54,238 - INFO - Checkpoint: 1,176,821/10,670,393 (11.0%) | 1.3 prompts/sec | errors: 0 | skipped: 1 | ETA: 1986.6h
2026-03-18 18:33:09,081 - INFO - Checkpoint: 1,177,821/10,670,393 (11.0%) | 1.3 prompts/sec | errors: 0 | skipped: 1 | ETA: 1962.0h
2026-03-18 18:45:22,353 - INFO - Checkpoint: 1,178,821/10,670,393 (11.0%) | 1.4 prompts/sec | errors: 0 | skipped: 3 | ETA: 1952.3h
2026-03-18 18:57:43,692 - INFO - Checkpoint: 1,179,821/10,670,393 (11.1%) | 1.4 prompts/sec | errors: 0 | skipped: 4 | ETA: 1952.7h
2026-03-18 19:10:06,083 - INFO - Checkpoint: 1,180,821/10,670,393 (11.1%) | 1.3 prompts/sec | errors: 0 | skipped: 5 | ETA: 1953.4h
2026-03-18 19:22:32,986 - INFO - Checkpoint: 1,181,821/10,670,393 (11.1%) | 1.3 prompts/sec | errors: 0 | skipped: 6 | ETA: 1955.7h
2026-03-18 19:34:54,382 - INFO - Checkpoint: 1,182,821/10,670,393 (11.1%) | 1.3 prompts/sec | errors: 0 | skipped: 7 | ETA: 1955.3h
2026-03-18 19:47:07,059 - INFO - Checkpoint: 1,183,821/10,670,393 (11.1%) | 

## 7. Post-Run Diagnostics

Run this after the processing completes (or after a crash) to inspect the results file.

In [ ]:
if os.path.exists(FULL_RESULTS_FILE):
    # Count rows correctly (handles quoted multiline fields)
    total_rows = count_csv_rows(FULL_RESULTS_FILE)

    # Read just the references_past column for stats
    ref_col = pd.read_csv(FULL_RESULTS_FILE, usecols=['references_past'])
    counts = ref_col['references_past'].value_counts()

    print(f"Results file: {FULL_RESULTS_FILE}")
    print(f"Total rows: {total_rows:,}")
    print()
    for val, cnt in counts.items():
        pct = cnt / total_rows * 100
        print(f"  {val:10s}: {cnt:>12,} ({pct:5.1f}%)")

    # Check for problems
    error_count = counts.get('error', 0)
    skipped_count = counts.get('skipped', 0)
    dup_headers = (ref_col['references_past'] == 'references_past').sum()

    print()
    if dup_headers > 0:
        print(f"  WARNING: {dup_headers} duplicate header(s) — run cell 5 to fix")
    if error_count > 0:
        print(f"  WARNING: {error_count:,} errors — run cell 5 to trim, then re-run cell 6")
    if skipped_count > 0:
        print(f"  INFO: {skipped_count:,} skipped (invalid prompts — too short, URLs, or numbers)")
    if error_count == 0 and dup_headers == 0:
        print("  File is clean!")

    # Source dataset comparison
    source_df = pd.read_csv('data/prompts.csv', usecols=[0])
    source_total = source_df.iloc[:, 0].dropna().count()
    print(f"\n  Source prompts: {source_total:,}")
    print(f"  Processed:     {total_rows:,}")
    print(f"  Remaining:     {source_total - total_rows:,}")
else:
    print(f"No results file found at {FULL_RESULTS_FILE}")